# LangGraph Dialogue System Simulation

This notebook runs the new `maestro_langgraph` module for testing.

In order to run this notebook from browser, run in your terminal :

`jupyter notebook src/maestro_langgraph/langgraph_simulation.ipynb`



## 1. Setup

In [1]:
import sys
from pathlib import Path

# Add maestro_langgraph to path
notebook_dir = Path.cwd()
module_dir = notebook_dir / "maestro_langgraph"

if module_dir.exists():
    sys.path.insert(0, str(notebook_dir))
    print(f"Added to path: {notebook_dir}")
else:
    print(f"Module not found at {module_dir}")
    print(f"Current dir: {notebook_dir}")

Added to path: /Users/yehor_kuzmych/maestro_langgraph/src/maestro_langgraph


In [2]:
# Check available backends
import subprocess
import urllib.request

def check_ollama():
    """Check if Ollama is running."""
    try:
        result = subprocess.run(["ollama", "list"], capture_output=True, text=True, timeout=5)
        print("✓ Ollama available")
        print(result.stdout)
        return True
    except Exception as e:
        print(f"✗ Ollama not available: {e}")
        return False

def check_lmstudio():
    """Check if LM Studio server is running."""
    try:
        req = urllib.request.Request("http://localhost:1234/v1/models", method="GET")
        with urllib.request.urlopen(req, timeout=5) as response:
            print("✓ LM Studio available at localhost:1234")
            return True
    except Exception as e:
        print(f"✗ LM Studio not available: {e}")
        return False

print("Checking available backends...")
print("-" * 50)
ollama_ok = check_ollama()
print()
lmstudio_ok = check_lmstudio()
print("-" * 50)

if ollama_ok:
    print("\n→ To use Ollama: BACKEND = 'ollama'")
if lmstudio_ok:
    print("→ To use LM Studio: BACKEND = 'lmstudio'")
if not ollama_ok and not lmstudio_ok:
    print("\n  No backend available!")
    print("   Start Ollama: ollama serve")
    print("   Or start LM Studio server")

Checking available backends...
--------------------------------------------------
✓ Ollama available
NAME                   ID              SIZE      MODIFIED    
qwen2.5:7b             845dbda0ea48    4.7 GB    10 days ago    
qwen2.5:3b             357c53fb659c    1.9 GB    3 weeks ago    
deepseek-coder:6.7b    ce298d984115    3.8 GB    5 weeks ago    
qwen2.5:0.5b           a8b0c5157701    397 MB    6 weeks ago    


✗ LM Studio not available: <urlopen error [Errno 61] Connection refused>
--------------------------------------------------

→ To use Ollama: BACKEND = 'ollama'


## 2. Import Module

In [3]:
from maestro_langgraph.state import DialogueState, Intent, Emotion, RobotStatus, get_prosody_for_emotion
from maestro_langgraph.chains import DialogueChains
from maestro_langgraph.graph import process_message, set_chains, build_dialogue_graph

print("✓ Imported maestro_langgraph module")
print(f"  Intents: {[i.value for i in Intent]}")
print(f"  Emotions: {[e.value for e in Emotion]}")
print(f"  Robot Status: {[s.value for s in RobotStatus]}")

✓ Imported maestro_langgraph module
  Intents: ['greeting', 'farewell', 'question', 'acknowledgment', 'general']
  Emotions: ['neutral', 'happy', 'sad', 'anger', 'surprise']
  Robot Status: ['free', 'busy', 'problem']


## 3. Initialize Chains

Choose your backend below:
- `"ollama"` - Uses Ollama with specified model (default: qwen2:0.5b)
- `"lmstudio"` - Uses LM Studio with currently loaded model

In [4]:
# ============================================
# CONFIGURE YOUR BACKEND HERE
# ============================================
BACKEND = "ollama"  # "ollama" or "lmstudio"
MODEL = "qwen2.5:7b"  # For Ollama only (LM Studio uses loaded model)
# ============================================

try:
    chains = DialogueChains(
        model_name=MODEL,
        backend=BACKEND,
    )
    set_chains(chains)
    print(f"✓ Initialized DialogueChains")
    print(f"  Backend: {BACKEND}")
    if BACKEND == "ollama":
        print(f"  Model: {MODEL}")
    else:
        print(f"  Model: (using LM Studio's loaded model)")
    print(f"  DuckDuckGo search: enabled")
except Exception as e:
    print(f" Failed to initialize: {e}")
    if BACKEND == "ollama":
        print(f"\nMake sure Ollama is running and model '{MODEL}' is pulled")
        print("  ollama serve")
        print(f"  ollama pull {MODEL}")
    else:
        print("\nMake sure LM Studio server is running on port 1234")
        print("  1. Open LM Studio")
        print("  2. Load a model")
        print("  3. Start the server (Developer tab)")

✓ Initialized DialogueChains
  Backend: ollama
  Model: qwen2.5:7b
  DuckDuckGo search: enabled


## 4. Test Individual Components

In [5]:
# Test intent classification
test_messages = [
    "Hello there!",
    "What is photosynthesis?",
    "Goodbye!",
    "Yes, that sounds good",
    "Tell me about soil moisture",
]

print("Testing Intent Classification:")
print("-" * 50)
for msg in test_messages:
    intent = chains.classify_intent(msg)
    print(f"  '{msg}' → {intent}")

Testing Intent Classification:
--------------------------------------------------
  'Hello there!' → greeting
  'What is photosynthesis?' → question
  'Goodbye!' → farewell
  'Yes, that sounds good' → acknowledgment
  'Tell me about soil moisture' → question


In [6]:
# Test web search
print("Testing DuckDuckGo Search:")
print("-" * 50)

query = "Who is Macron?"
print(f"Query: {query}")
results = chains.search_web(query)
print(f"Results (first 500 chars):\n{results[:500]}...")

Testing DuckDuckGo Search:
--------------------------------------------------
Query: Who is Macron?
Results (first 500 chars):
Emmanuel Macron is a French banker and politician who was elected president of France in 2017. Macron was the first person in the history of the Fifth Republic to win the presidency without the backing of either the Socialists or the Gaullists, and he was France's youngest head of state since Napoleon. President Emmanuel Macron , speaking at the Munich Security Conference, stressed European "fortitude." Earlier, Chancellor Friedrich Merz of Germany criticized President Trump's rapid ... France, ...


In [7]:
# Test response generation
print("Testing Response Generation:")
print("-" * 50)

response = chains.generate_response(
    message="Hello, how are you?",
    emotion="happy",
    history=[]
)
print(f"Response: {response}")

Testing Response Generation:
--------------------------------------------------
Response: Hello! I'm doing great, thanks for asking. How about you? It's so nice to chat with a happy face! 😊


In [8]:
# Test emotion detection
print("Testing Emotion Detection:")
print("-" * 50)

emotion = chains.detect_emotion(
    user_message="Didnt know the tomatos grow so fast",
    robot_response=""
)
print(f"Detected emotion: {emotion}")
print(f"Prosody: {get_prosody_for_emotion(emotion)}")

Testing Emotion Detection:
--------------------------------------------------
Detected emotion: surprise
Prosody: (200, 80, 65)


In [9]:
# Test emotion detection
print("Testing Emotion Detection:")
print("-" * 50)

emotion = chains.detect_emotion(
    user_message="I hate rain",
    robot_response=""
)
print(f"Detected emotion: {emotion}")
print(f"Prosody: {get_prosody_for_emotion(emotion)}")

Testing Emotion Detection:
--------------------------------------------------
Detected emotion: sad
Prosody: (80, 80, 35)


In [10]:
# Test emotion detection
print("Testing Emotion Detection:")
print("-" * 50)

emotion = chains.detect_emotion(
    user_message="I bought new seeds today, but unfortunately lost them in the subway",
    robot_response=""
)
print(f"Detected emotion: {emotion}")
print(f"Prosody: {get_prosody_for_emotion(emotion)}")

Testing Emotion Detection:
--------------------------------------------------
Detected emotion: sad
Prosody: (80, 80, 35)


In [11]:
# Test emotion detection
print("Testing Emotion Detection:")
print("-" * 50)

emotion = chains.detect_emotion(
    user_message="Looking forward for the moment when I can harvest my own cucumbers!",
    robot_response=""
)
print(f"Detected emotion: {emotion}")
print(f"Prosody: {get_prosody_for_emotion(emotion)}")

Testing Emotion Detection:
--------------------------------------------------
Detected emotion: happy
Prosody: (160, 140, 65)


## 5. Run Full Dialogue Graph

In [12]:
def chat(message: str, history: list = None, voice_emotion: str = "neutral"):
    """Process a message through the dialogue graph."""
    print(f"\n{'='*60}")
    print(f"Human: {message}")
    print(f"{'='*60}")

    result = process_message(
        message=message,
        voice_emotion=voice_emotion,
        history=history or [],
    )

    print(f"\nIntent: {result.get('intent', 'unknown')}")
    if result.get('search_context'):
        print(f"Search: Yes (found context)")
    print(f"\nPlantroid: {result.get('response', '')}")
    print(f"   Emotion: {result.get('response_emotion', 'neutral')}")
    print(f"   Prosody: {result.get('prosody', (150, 100, 45))}")

    return result

In [13]:
# Test greeting
result = chat("Hello Plantroid!")


Human: Hello Plantroid!

Intent: greeting

Plantroid: Hello there! Nice to meet you. How can I help you with your plants today?
   Emotion: neutral
   Prosody: (150, 100, 45)


In [ ]:
# Test question (triggers web search)
result = chat("What is the best soil for tomatoes?")


Human: What is the best soil for tomatoes?


In [ ]:
# Test factual question with search
result = chat("Who is Albert Einstein?")

In [ ]:
# Test farewell
result = chat("Goodbye!")

## 6. Multi-turn Conversation

In [17]:
# Multi-turn conversation with history
history = []

print("\n" + "="*60)
print("MULTI-TURN CONVERSATION")
print("="*60)

messages = [
    "Hi there!",
    "What's your name?",
    "Can you help me with my plants?",
    "What is the ideal temperature for succulents?",
    "Thanks for your help!",
]

for msg in messages:
    result = process_message(
        message=msg,
        history=history,
    )

    print(f"\nHuman: {msg}")
    print(f"Plantroid: {result.get('response', '')}")

    # Update history
    history = result.get('conversation_history', [])


MULTI-TURN CONVERSATION

Human: Hi there!
 Plantroid: Hi there! 😊 Hi there! 🌟

Human: What's your name?
 Plantroid: Hi there! 🌟 How can I assist you today? 😊

Human: Can you help me with my plants?
 Plantroid: Hello! I'm here to help with your plants. 🌟 Feel free to ask any questions or let me know when you're ready for a chat about your plants. 😊

Human: What is the ideal temperature for succulents?
 Plantroid: Hello! 🌟 I'm here to help with your succulent care questions. Feel free to ask if you have any specific questions or concerns about temperature ranges for your plants. 😊

Human: Thanks for your help!
 Plantroid: Hello! Glad to help with your succulent care! Just remember, different types of succulents have varying temperature ranges. Make sure you're setting their needs in mind. 😊 Enjoy your plants! 🌟


In [18]:
# View conversation history
print("Conversation History:")
print("-" * 40)
for role, msg in history:
    print(f"{role}: {msg}")

Conversation History:
----------------------------------------
Human: Hi there!
Plantroid: Hi there! 😊 Hi there! 🌟
Human: What's your name?
Plantroid: Hi there! 🌟 How can I assist you today? 😊
Human: Can you help me with my plants?
Plantroid: Hello! I'm here to help with your plants. 🌟 Feel free to ask any questions or let me know when you're ready for a chat about your plants. 😊
Human: What is the ideal temperature for succulents?
Plantroid: Hello! 🌟 I'm here to help with your succulent care questions. Feel free to ask if you have any specific questions or concerns about temperature ranges for your plants. 😊
Human: Thanks for your help!
Plantroid: Hello! Glad to help with your succulent care! Just remember, different types of succulents have varying temperature ranges. Make sure you're setting their needs in mind. 😊 Enjoy your plants! 🌟


In [19]:
# Test FREE state - normal conversation flow
print("="*60)
print("TEST: Robot is FREE (normal flow)")
print("="*60)

result = process_message(
    message="What's the best way to water succulents?",
    robot_busy=False,
    notifications={},  # No problems
)

print(f"Robot Status: {result.get('robot_status')}")
print(f"Intent: {result.get('intent')}")
print(f"Response: {result.get('response')}")
print(f"Emotion: {result.get('response_emotion')}")

TEST: Robot is FREE (normal flow)
Robot Status: free
Intent: general
Response: Watering succulents is simple! Aim for 1-2 inches of water per week during the hottest months. Regularly check soil moisture and adjust accordingly to avoid burning. Enjoy your plants!
Emotion: happy


In [20]:
# Test PROBLEM state - sensor notifications pending
print("="*60)
print("TEST: Robot has PROBLEM notifications")
print("="*60)

# Simulate sensor notifications (like original MAESTRO)
notifications = {
    "soil_moisture": ["low", "high"],
    "temperature": ["28°C", "medium"],
}

result = process_message(
    message="Hi there!",
    robot_busy=False,
    notifications=notifications,  # Has problems to announce!
)

print(f"Robot Status: {result.get('robot_status')}")
print(f"Response: {result.get('response')}")
print(f"Emotion: {result.get('response_emotion')} (should be 'sad' for concern)")
print(f"Should end early: {result.get('should_end_early')} (False = allow follow-up)")

TEST: Robot has PROBLEM notifications
Robot Status: problem
Response: Dear user, I've observed that the soil moisture level is currently low. It's important to monitor and ensure your plants receive enough water to thrive. Could you please check if there are any signs of underwatering or overwatering? If so, it could indicate a potential issue. Let me know how we can address this together.

Sensor issue to mention:
- Type: temperature
- Current value: 28.0°C
- Optimal range: 20.0-25.0°C
- Problem: too high
- Severity: high

Suggested action: turn on a fan or open windows for better air circulation
Expected outcome: get it back to optimal levels (20.0-25.0°C)

I noticed that the soil moisture level is currently low, which could indicate underwatering or overwatering issues. To address this, I've suggested using a fan or opening windows for improved air circulation and water drainage. Let's ensure your plants receive enough water to stay healthy.

By the way, please check if there are an

In [21]:
# Test BUSY state - robot is busy with a task
print("="*60)
print("TEST: Robot is BUSY")
print("="*60)

result = process_message(
    message="Hello, can you help me?",
    robot_busy=True,  # Robot is busy!
    notifications={},
)

print(f"Robot Status: {result.get('robot_status')}")
print(f"Response: {result.get('response')}")
print(f"Emotion: {result.get('response_emotion')}")
print(f"Should end early: {result.get('should_end_early')}")

TEST: Robot is BUSY
Robot Status: busy
Response: Hello! I'm here to assist you with whatever tasks you have. While I can't directly answer your questions or provide immediate help, I am always here to offer advice and support when needed. Please let me know if there's anything specific you'd like assistance with next.
Emotion: neutral
Should end early: True


In [22]:
# Visualize the dialogue graph structure
try:
    from langgraph.graph import StateGraph

    print("LangGraph Dialogue Flow (with Busy/Problem Check):")
    print("="*60)
    print("""
    START
      │
      ▼
    ┌─────────────────┐
    │  check_context  │  ← Check busy/problem status
    └────────┬────────┘
             │
      ┌──────┴──────┐
      │             │
      ▼             ▼
   [BUSY]       [FREE/PROBLEM]
      │             │
      │             ▼
      │     ┌─────────────────┐
      │     │ classify_intent │  ← LLM classifies intent
      │     └────────┬────────┘
      │              │
      │              ▼
      │     ┌─────────────────┐
      │     │  check_search   │  ← DuckDuckGo if needed
      │     └────────┬────────┘
      │              │
      └──────┬───────┘
             │
             ▼
    ┌─────────────────┐
    │generate_response│  ← LLM generates response
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │determine_emotion│  ← Emotion based on context
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │ update_history  │  ← Store conversation
    └────────┬────────┘
             │
             ▼
           END
             
    Context-aware emotions:
    - BUSY → neutral (apologetic)
    - PROBLEM → sad (concerned)
    - FREE → LLM-determined
    """)

except Exception as e:
    print(f"Could not visualize: {e}")

LangGraph Dialogue Flow (with Busy/Problem Check):

    START
      │
      ▼
    ┌─────────────────┐
    │  check_context  │  ← Check busy/problem status
    └────────┬────────┘
             │
      ┌──────┴──────┐
      │             │
      ▼             ▼
   [BUSY]       [FREE/PROBLEM]
      │             │
      │             ▼
      │     ┌─────────────────┐
      │     │ classify_intent │  ← LLM classifies intent
      │     └────────┬────────┘
      │              │
      │              ▼
      │     ┌─────────────────┐
      │     │  check_search   │  ← DuckDuckGo if needed
      │     └────────┬────────┘
      │              │
      └──────┬───────┘
             │
             ▼
    ┌─────────────────┐
    │generate_response│  ← LLM generates response
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │determine_emotion│  ← Emotion based on context
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │ update_history

## 7. Interactive Chat

In [23]:
# Interactive chat loop
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("Widgets not available. Using text input.")

if WIDGETS_AVAILABLE:
    output = widgets.Output()
    text_input = widgets.Text(
        placeholder='Talk to Plantroid...',
        layout=widgets.Layout(width='70%')
    )
    send_btn = widgets.Button(description='Send', button_style='primary')
    clear_btn = widgets.Button(description='Clear', button_style='warning')

    chat_history = []

    def on_send(_):
        global chat_history
        if text_input.value:
            with output:
                result = process_message(
                    message=text_input.value,
                    history=chat_history,
                )
                print(f"\nYou: {text_input.value}")
                print(f"Plantroid: {result.get('response', '')} [{result.get('response_emotion', '')}]")
                chat_history = result.get('conversation_history', [])
            text_input.value = ''

    def on_clear(_):
        global chat_history
        chat_history = []
        with output:
            clear_output()
            print("Chat cleared.")

    send_btn.on_click(on_send)
    clear_btn.on_click(on_clear)
    text_input.on_submit(lambda _: on_send(None))

    print(" Chat with Plantroid (LangGraph)")
    print("All responses generated by LLM - no hardcoded flows!")
    display(widgets.HBox([text_input, send_btn, clear_btn]))
    display(output)

 Chat with Plantroid (LangGraph)
All responses generated by LLM - no hardcoded flows!


/var/folders/5y/wlzg0qqx2wz8yjp7kvnkdg_m0000gn/T/ipykernel_73626/3614088516.py:43: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  text_input.on_submit(lambda _: on_send(None))


Output()

## 8. Visualize Graph

In [24]:
# Visualize the dialogue graph structure
try:
    from langgraph.graph import StateGraph

    print("LangGraph Dialogue Flow:")
    print("="*50)
    print("""
    START
      │
      ▼
    ┌─────────────────┐
    │ classify_intent │  ← LLM classifies user intent
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │  check_search   │  ← DuckDuckGo if needed
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │generate_response│  ← LLM generates response
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │determine_emotion│  ← LLM detects emotion
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │ update_history  │  ← Store conversation
    └────────┬────────┘
             │
             ▼
           END
    """)

except Exception as e:
    print(f"Could not visualize: {e}")

LangGraph Dialogue Flow:

    START
      │
      ▼
    ┌─────────────────┐
    │ classify_intent │  ← LLM classifies user intent
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │  check_search   │  ← DuckDuckGo if needed
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │generate_response│  ← LLM generates response
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │determine_emotion│  ← LLM detects emotion
    └────────┬────────┘
             │
             ▼
    ┌─────────────────┐
    │ update_history  │  ← Store conversation
    └────────┬────────┘
             │
             ▼
           END
    


## 9. Performance Test

In [25]:
import time

# Measure response time
test_messages = [
    "Hello!",
    "How are you?",
    "What is the weather like?",
]

print("Performance Test:")
print("-" * 50)

times = []
for msg in test_messages:
    start = time.time()
    result = process_message(message=msg)
    elapsed = time.time() - start
    times.append(elapsed)
    print(f"'{msg}' → {elapsed:.2f}s")

print(f"\nAverage response time: {sum(times)/len(times):.2f}s")
print(f"Backend: {BACKEND}")
if BACKEND == "ollama":
    print(f"Model: {MODEL}")

Performance Test:
--------------------------------------------------
'Hello!' → 0.89s
'How are you?' → 0.48s
'What is the weather like?' → 1.73s

Average response time: 1.03s
Backend: ollama
Model: qwen2.5:0.5b
